# ========== Import Necessary Libraries ==========

In [ ]:
# Standard library imports
import os
import random
import hashlib
import numpy as np
import math
import shutil

# Image processing and visualization imports
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# Machine learning and deep learning imports
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import models, layers, optimizers, losses, metrics, applications
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model, Model, model_from_json
from tensorflow.keras.layers import Conv2D, BatchNormalization, GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.callbacks import LearningRateScheduler, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.applications import Xception
from tensorflow.keras.applications.xception import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix

# ========== Data Preparation and Preprocessing ==========

In [ ]:
# Step 1: Environment Setup and Data Acquisition

# Set environment variables for Kaggle API
# ضعي مفتاح Kaggle في متغيرات البيئة قبل التشغيل — لا يُكتب في الكود
# os.environ['KAGGLE_USERNAME'] = ...
# os.environ['KAGGLE_KEY'] = ...

# Download and unzip dataset from Kaggle
!kaggle datasets download -d reemaalfaleh/detection-dataset
!unzip detection-dataset.zip


Streaming output truncated to the last 5000 lines.
  inflating: COVID-19_Radiography_Dataset/Viral Pneumonia/images/1309.jpeg  
  inflating: COVID-19_Radiography_Dataset/Viral Pneumonia/images/131.jpeg  
  inflating: COVID-19_Radiography_Dataset/Viral Pneumonia/images/1310.jpeg  
  inflating: COVID-19_Radiography_Dataset/Viral Pneumonia/images/1311.jpeg  
  inflating: COVID-19_Radiography_Dataset/Viral Pneumonia/images/1312.jpeg  
  inflating: COVID-19_Radiography_Dataset/Viral Pneumonia/images/1313.jpeg  
  inflating: COVID-19_Radiography_Dataset/Viral Pneumonia/images/1314.jpeg  
  inflating: COVID-19_Radiography_Dataset/Viral Pneumonia/images/1315.jpeg  
  inflating: COVID-19_Radiography_Dataset/Viral Pneumonia/images/1316.jpeg  
  inflating: COVID-19_Radiography_Dataset/Viral Pneumonia/images/1317.jpeg  
  inflating: COVID-19_Radiography_Dataset/Viral Pneumonia/images/1318.jpeg  
  inflating: COVID-19_Radiography_Dataset/Viral Pneumonia/images/1319.jpeg  
  inflating: COVID-19_Radi

In [ ]:
# Step 2: Initial Data Inspection (orignal data)

images_COVID = len(os.listdir('/content/COVID-19_Radiography_Dataset/COVID/images'))
images_Lung_Opacity = len(os.listdir('/content/COVID-19_Radiography_Dataset/Lung_Opacity/images'))
images_Normal = len(os.listdir('/content/COVID-19_Radiography_Dataset/Normal/images'))
images_ViralPneumonia = len(os.listdir('/content/COVID-19_Radiography_Dataset/Viral Pneumonia/images'))
print(f"Covid class images: {images_COVID} | Normal class images: {images_Normal} | Lung class images: {images_Lung_Opacity} | Viral class images: {images_ViralPneumonia}")

Covid class images: 7616 | Normal class images: 10192 | Lung class images: 6012 | Viral class images: 5345


In [ ]:
# Step 3: Duplicate Removal

# Function to ensure a directory exists
def ensure_directory_exists(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

# Function to calculate file hash, used for identifying duplicates
def file_hash(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

# Function to remove duplicate images based on file hash
def remove_duplicates(directory):
    hashes = {}
    for subdir, dirs, files in os.walk(directory):
        for file in files:
            path = os.path.join(subdir, file)
            file_hash_value = file_hash(path)
            if file_hash_value in hashes:
                os.remove(path)
            else:
                hashes[file_hash_value] = path
    print("Duplicate removal process completed.")

# Paths
original_data_path = '/content/COVID-19_Radiography_Dataset'

# Remove Duplicates
remove_duplicates(original_data_path)




Duplicate removal process completed.


In [ ]:
#After Duplicate removal process
images_COVID = len(os.listdir('/content/COVID-19_Radiography_Dataset/COVID/images'))
images_Lung_Opacity = len(os.listdir('/content/COVID-19_Radiography_Dataset/Lung_Opacity/images'))
images_Normal = len(os.listdir('/content/COVID-19_Radiography_Dataset/Normal/images'))
images_ViralPneumonia = len(os.listdir('/content/COVID-19_Radiography_Dataset/Viral Pneumonia/images'))
print(f"Covid class images: {images_COVID} | Normal class images: {images_Normal} | Lung class images: {images_Lung_Opacity} | Viral class images: {images_ViralPneumonia}")

Covid class images: 7230 | Normal class images: 10191 | Lung class images: 6012 | Viral class images: 5338


In [ ]:
# Step 4: Conversion to Uniform Format

# Function to convert images to PNG format and count them before and after conversion
def convert_to_png_and_count(directory):
    counts_before_conversion = {}
    for class_folder in os.listdir(directory):
        class_path = os.path.join(directory, class_folder)
        counts_before_conversion[class_folder] = len([file for file in os.listdir(class_path) if file.lower().endswith('.png')])

    for subdir, _, files in os.walk(directory):
        for file in files:
            if not file.lower().endswith('.png'):
                path = os.path.join(subdir, file)
                image = Image.open(path)
                png_path = os.path.splitext(path)[0] + '.png'
                image.save(png_path, 'PNG')
                os.remove(path)

    # Count images after conversion
    counts_after_conversion = {}
    for class_folder in os.listdir(directory):
        class_path = os.path.join(directory, class_folder)
        counts_after_conversion[class_folder] = len(os.listdir(class_path))

    return counts_before_conversion, counts_after_conversion

    # Convert to PNG and count before and after
counts_before_conversion, counts_after_conversion = convert_to_png_and_count(original_data_path)
print("\nNumber of images per class before and after PNG conversion:")
for class_name in counts_after_conversion.keys():
    print(f"{class_name}: Before conversion: {counts_before_conversion[class_name]}, After conversion: {counts_after_conversion[class_name]}")

# Convert images to PNG and remove originals
# Returns counts of images before and after conversion


Number of images per class before and after PNG conversion:
COVID: Before conversion: 0, After conversion: 1
Viral Pneumonia: Before conversion: 0, After conversion: 1
Normal: Before conversion: 0, After conversion: 1
Lung_Opacity: Before conversion: 0, After conversion: 1


In [ ]:
#After Duplicate removal process and conversion
images_COVID = len(os.listdir('/content/COVID-19_Radiography_Dataset/COVID/images'))
images_Lung_Opacity = len(os.listdir('/content/COVID-19_Radiography_Dataset/Lung_Opacity/images'))
images_Normal = len(os.listdir('/content/COVID-19_Radiography_Dataset/Normal/images'))
images_ViralPneumonia = len(os.listdir('/content/COVID-19_Radiography_Dataset/Viral Pneumonia/images'))
print(f"Covid class images: {images_COVID} | Normal class images: {images_Normal} | Lung class images: {images_Lung_Opacity} | Viral class images: {images_ViralPneumonia}")

Covid class images: 7230 | Normal class images: 10191 | Lung class images: 6012 | Viral class images: 5338


In [ ]:
# Step 5: Balancing the Dataset

# Function to balance classes to the same number of images
  # Balances the number of images across classes by sampling

def ensure_directory_exists(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def balance_classes(source_dir, target_dir, desired_samples):
    classes = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
    for class_name in classes:
        src_class_dir = os.path.join(source_dir, class_name, 'images')  # Adjusted to include 'images' if your structure requires
        tgt_class_dir = os.path.join(target_dir, class_name)
        ensure_directory_exists(tgt_class_dir)

        images = [img for img in os.listdir(src_class_dir) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]  # Ensure only image files are considered
        selected_images = random.sample(images, min(len(images), desired_samples)) if len(images) > desired_samples else images

        for img_name in selected_images:
            src_path = os.path.join(src_class_dir, img_name)
            dst_path = os.path.join(tgt_class_dir, img_name)
            shutil.copy(src_path, dst_path)

        print(f"Class {class_name}: Balanced with {len(selected_images)} images.")

# Main execution
original_data_path = '/content/COVID-19_Radiography_Dataset'
balanced_data_path = '/content/balanced_images'

# Remove duplicates (Assuming you've handled it previously or using a different snippet)

# Ensure the target directory for balanced images exists
ensure_directory_exists(balanced_data_path)

# Balance each class to have the desired number of samples and save them in a new directory
desired_samples = 5300
balance_classes(original_data_path, balanced_data_path, desired_samples)

print("Balancing complete. Balanced images are saved in the 'balanced_images' directory.")


Class COVID: Balanced with 5300 images.
Class Lung_Opacity: Balanced with 5300 images.
Class Normal: Balanced with 5300 images.
Class Viral Pneumonia: Balanced with 5300 images.
Balancing complete. Balanced images are saved in the 'balanced_images' directory.


In [ ]:
# Step 6: Dataset Splitting

# Function to split dataset into training, validation, and test sets
 # Splits the balanced dataset into train, validation, and test sets

def ensure_directory_exists(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def split_dataset(balanced_data_path, split_data_path, train_ratio=0.7, validation_ratio=0.15):
    test_ratio = 1 - train_ratio - validation_ratio
    classes = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']

    # Ensure base directories for split dataset exist
    for phase in ['train', 'validation', 'test']:
        ensure_directory_exists(os.path.join(split_data_path, phase))

    # Iterate through each class and perform splitting
    for class_name in classes:
        class_path = os.path.join(balanced_data_path, class_name)
        images = os.listdir(class_path)
        # Split dataset
        train_images, test_val_images = train_test_split(images, test_size=test_ratio, random_state=42)
        validation_images, test_images = train_test_split(test_val_images, test_size=test_ratio/(test_ratio + validation_ratio), random_state=42)

        # Copy images to their respective directories
        for img_name in train_images:
            src = os.path.join(class_path, img_name)
            dst = os.path.join(split_data_path, 'train', class_name, img_name)
            ensure_directory_exists(os.path.dirname(dst))
            shutil.copy(src, dst)

        for img_name in validation_images:
            src = os.path.join(class_path, img_name)
            dst = os.path.join(split_data_path, 'validation', class_name, img_name)
            ensure_directory_exists(os.path.dirname(dst))
            shutil.copy(src, dst)

        for img_name in test_images:
            src = os.path.join(class_path, img_name)
            dst = os.path.join(split_data_path, 'test', class_name, img_name)
            ensure_directory_exists(os.path.dirname(dst))
            shutil.copy(src, dst)

        print(f"{class_name} split into train, validation, and test sets successfully.")

# Define your paths
balanced_data_path = '/content/balanced_images'
split_data_path = '/content/splitData'

# Perform the split
split_dataset(balanced_data_path, split_data_path, train_ratio=0.7, validation_ratio=0.15)

print("Dataset splitting complete.")


COVID split into train, validation, and test sets successfully.
Lung_Opacity split into train, validation, and test sets successfully.
Normal split into train, validation, and test sets successfully.
Viral Pneumonia split into train, validation, and test sets successfully.
Dataset splitting complete.


In [ ]:
COVID_trainData = len(os.listdir('/content/splitData/train/COVID'))
COVID_validationData = len(os.listdir('/content/splitData/validation/COVID'))
COVID_testData = len(os.listdir('/content/splitData/test/COVID'))
print(f"Train COVID images: {COVID_trainData} | validation COVID images: {COVID_validationData} | test COVID images: {COVID_testData}")

Lung_Opacity_trainData = len(os.listdir('/content/splitData/train/Lung_Opacity'))
Lung_Opacity_validationData = len(os.listdir('/content/splitData/validation/Lung_Opacity'))
Lung_Opacity_testData = len(os.listdir('/content/splitData/test/Lung_Opacity'))
print(f"Train Lung_Opacity images: {Lung_Opacity_trainData} | validation Lung_Opacity images: {Lung_Opacity_validationData} | test Lung_Opacity images: {Lung_Opacity_testData}")

Normal_trainData = len(os.listdir('/content/splitData/train/Normal'))
Normal_validationData = len(os.listdir('/content/splitData/validation/Normal'))
Normal_testData = len(os.listdir('/content/splitData/test/Normal'))
print(f"Train Normal images: {Normal_trainData} | validation Normal images: {Normal_validationData} | test Normal images: {Normal_testData}")

ViralPneumonia_trainData = len(os.listdir('/content/splitData/train/Viral Pneumonia'))
ViralPneumonia_validationData = len(os.listdir('/content/splitData/validation/Viral Pneumonia'))
ViralPneumonia_testData = len(os.listdir('/content/splitData/test/Viral Pneumonia'))
print(f"Train Viral Pneumonia images: {ViralPneumonia_trainData} | validation Viral Pneumonia images: {ViralPneumonia_validationData} | test Viral Pneumonia images: {ViralPneumonia_testData}")


Train COVID images: 4504 | validation COVID images: 397 | test COVID images: 399
Train Lung_Opacity images: 4504 | validation Lung_Opacity images: 397 | test Lung_Opacity images: 399
Train Normal images: 4504 | validation Normal images: 397 | test Normal images: 399
Train Viral Pneumonia images: 4504 | validation Viral Pneumonia images: 397 | test Viral Pneumonia images: 399


# ========== Model Definition and Training ==========

In [ ]:
# Load the Pre-Trained Xception Model
base_model = Xception(weights='imagenet', include_top=False, input_shape=(299, 299, 3))

# Enhanced Model Architecture
x = base_model.output
x = Conv2D(64, (3, 3), activation='relu')(x)
x = BatchNormalization()(x)
x = Conv2D(64, (3, 3), activation='relu')(x)
x = BatchNormalization()(x)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(2048, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
predictions = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Set layers to trainable
for layer in base_model.layers:
    layer.trainable = True

model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:

# Load your models
disease_model = load_model('xception_finetuned_model.h5')
xray_detector = load_model('xray_detector.h5')

# Function to check if an uploaded image is an X-ray
def is_xray_image(img_path):
    """Check if an image is an X-ray."""
    img = image.load_img(img_path, target_size=(299, 299))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    predictions = xray_detector.predict(img_array)
    return predictions[0, 0] > 0.5

# Function to preprocess images for model input
def preprocess_image(x):
    x /= 255.0
    x -= 0.5
    x *= 2.0
    return x


In [ ]:
# Setting up the image data generators
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_image,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest',
    brightness_range=[0.8, 1.2]
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_image)

# Data Generators
train_data_dir = os.path.join(split_data_path, 'train')
validation_data_dir = os.path.join(split_data_path, 'validation')
test_data_dir = os.path.join(split_data_path, 'test')


train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(299, 299),
    batch_size=32,
    class_mode='categorical'
)

validation_generator = test_datagen.flow_from_directory(
    validation_data_dir,
    target_size=(299, 299),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(299, 299),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 18016 images belonging to 4 classes.
Found 1588 images belonging to 4 classes.
Found 1596 images belonging to 4 classes.


In [ ]:

# Callbacks for training adjustments
callbacks_list = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint('model_best_weights.h5', monitor='val_loss', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=10, min_lr=0.00001, verbose=1)
]

# Model training
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=50,
    validation_data=validation_generator,
    validation_steps=len(validation_generator),
    callbacks=callbacks_list
)


Epoch 1/50
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8072 - loss: 1.5361

563/563 ━━━━━━━━━━━━━━━━━━━━ 1032s 2s/step - accuracy: 0.8759 - loss: 1.1255 - val_accuracy: 0.9320 - val_loss: 0.5846 - learning_rate: 1.0000e-04
Epoch 2/50
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9264 - loss: 0.5245

563/563 ━━━━━━━━━━━━━━━━━━━━ 871s 2s/step - accuracy: 0.9270 - loss: 0.4527 - val_accuracy: 0.9326 - val_loss: 0.3069 - learning_rate: 1.0000e-04
Epoch 3/50
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9354 - loss: 0.2949

563/563 ━━━━━━━━━━━━━━━━━━━━ 861s 2s/step - accuracy: 0.9364 - loss: 0.2743 - val_accuracy: 0.9547 - val_loss: 0.1751 - learning_rate: 1.0000e-04
Epoch 4/50
198/563 ━━━━━━━━━━━━━━━━━━━━ 9:03 1s/step - accuracy: 0.9487 - loss: 0.2157

# ========== Model Evaluation and Utilization ==========


In [ ]:

# Model Evaluation and Utilization
test_loss, test_accuracy = model.evaluate(test_generator)
predictions = model.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

# Classification Report and Confusion Matrix
cm = confusion_matrix(true_classes, predicted_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

print(classification_report(true_classes, predicted_classes, target_names=class_labels))

# Save the model
model.save('xception_finetuned_model.h5')

In [ ]:
def make_prediction(model, img_path, class_labels):
    """
    Predicts the class of a given image using the specified model.

    Args:
    model (tf.keras.Model): The pre-loaded deep learning model for classification.
    img_path (str): The path to the image file to classify.
    class_labels (list): A list of string labels for classification.

    Returns:
    str: A prediction label from `class_labels` or an error message.
    """
    # First, check if the image is an X-ray
    if is_xray_image(img_path):
        img = image.load_img(img_path, target_size=(299, 299))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        preprocessed_image = preprocess_input(img_array)
        predictions = model.predict(preprocessed_image)
        predicted_class_index = np.argmax(predictions, axis=1)
        predicted_class_label = class_labels[predicted_class_index[0]]
        return f"Predicted class: {predicted_class_label}"
    else:
        return "Uploaded image is not an X-ray."

# Example usage:
img_path = '/content/splitData/test/Lung_Opacity/Lung_Opacity-1002.png'  # Update this path as necessary
class_labels = ['COVID', 'Lung Opacity', 'Normal', 'Viral Pneumonia']
prediction = make_prediction(disease_model, img_path, class_labels)
print(prediction)


# ========== Plots for Checking ==========


In [ ]:
# To understand where the model is focusing, Grad-CAM can be helpful.

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, classifier_layer_names):
    # First, create a model that maps the input image to the activations
    # of the last conv layer as well as the output predictions
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    # Then, compute the gradient of the top predicted class for our input image
    # with respect to the activations of the last conv layer
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if preds.shape[-1] > 1:
            pred_index = tf.argmax(preds[0])
        else:
            pred_index = 0
        class_channel = preds[:, pred_index]

    # This is the gradient of the output neuron (top predicted or chosen)
    # with regard to the output feature map of the last conv layer
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # This is a vector where each entry is the mean intensity of the gradient
    # over a specific feature map channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # We multiply each channel in the feature map array
    # by "how important this channel is" with regard to the top predicted class
    # then sum all the channels to obtain the heatmap class activation
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # For visualization purpose, we will also normalize the heatmap between 0 & 1
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Preprocess your image and predict
# Ensure img_path is updated to point to a valid image file
img_path = '/content/splitData/test/Lung_Opacity/Lung_Opacity-1004.png'
img = load_img(img_path, target_size=(299, 299))
img_array = img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

# Update 'last_conv_layer_name' to the name of the last convolutional layer in your model
last_conv_layer_name = 'block14_sepconv2_act'
classifier_layer_names = []  # This is not used in this simplified version but can be useful in other contexts

# Generate the heatmap
heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name, classifier_layer_names)

# Display Heatmap
plt.matshow(heatmap)
plt.show()


In [ ]:
# Check class distribution in the training set
train_classes = train_generator.classes
class_counts = np.unique(train_classes, return_counts=True)[1]
class_labels = list(train_generator.class_indices.keys())

print("Training set class distribution:")
for label, count in zip(class_labels, class_counts):
    print(f"{label}: {count}")

# Optionally, for a visual representation, you can plot this distribution
plt.bar(class_labels, class_counts)
plt.title('Class Distribution in Training Set')
plt.xlabel('Class')
plt.ylabel('Number of images')
plt.show()


In [ ]:
# Check class distribution in the validation set
validation_classes = validation_generator.classes
class_counts = np.unique(validation_classes, return_counts=True)[1]
class_labels = list(validation_generator.class_indices.keys())

print("validation set class distribution:")
for label, count in zip(class_labels, class_counts):
    print(f"{label}: {count}")

# Optionally, for a visual representation, you can plot this distribution
plt.bar(class_labels, class_counts)
plt.title('Class Distribution in validation Set')
plt.xlabel('Class')
plt.ylabel('Number of images')
plt.show()


In [ ]:
# Check class distribution in the test set
test_classes = test_generator.classes
class_counts = np.unique(test_classes, return_counts=True)[1]
class_labels = list(test_generator.class_indices.keys())

print("test set class distribution:")
for label, count in zip(class_labels, class_counts):
    print(f"{label}: {count}")

# Optionally, for a visual representation, you can plot this distribution
plt.bar(class_labels, class_counts)
plt.title('Class Distribution in test Set')
plt.xlabel('Class')
plt.ylabel('Number of images')
plt.show()


In [ ]:
model.summary()